In [24]:
!pip install rasterio
!pip install tifffile
!pip install imagecodecs


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
import rasterio
import numpy as np

def inspect_tiff(path):
    with rasterio.open(path) as src:
        dtype_str = src.dtypes[0]        # string like "float32"
        dtype = np.dtype(dtype_str)      # real numpy dtype object
        nodata = src.nodatavals[0]
        compression = src.compression.name if src.compression else None
        
        block_shape = src.block_shapes[0]
        is_tiled = block_shape != (src.height, src.width)

        return {
            "width": src.width,
            "height": src.height,
            "totalPixels": src.width * src.height,
            "bands": src.count,
            "dtype": dtype_str,
            "bitsPerSample": dtype.itemsize * 8,
            "bytesPerSample": dtype.itemsize,
            "compression": compression,
            "isTiled": is_tiled,
            "tileShape": block_shape,
            "noData": nodata,
            "transform": src.transform,
            "crs": src.crs.to_string() if src.crs else None,
        }

info = inspect_tiff(
    r"C:/Users/Aaron McLean/ccsvi-dashboard/public/data/Rasters/Hawaii_Category4_MOM_Inundation_HighTide.tif"
    #r"C:/Users/Aaron McLean/ccsvi-dashboard/public/data/Rasters/lw_hi.tif"
)
info


{'width': 62528,
 'height': 41519,
 'totalPixels': 2596100032,
 'bands': 1,
 'dtype': 'uint8',
 'bitsPerSample': 8,
 'bytesPerSample': 1,
 'compression': 'lzw',
 'isTiled': True,
 'tileShape': (128, 128),
 'noData': 255.0,
 'transform': Affine(8.129955900000169e-05, 0.0, -159.8770750988412,
        0.0, -8.129955900000175e-05, 22.281202920435213),
 'crs': 'GEOGCS["NAD83",DATUM["North American Datum 1983",SPHEROID["GRS 1980",6378137,298.257222101004]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]'}

In [49]:
info2 = inspect_tiff(
    #r"C:/Users/Aaron McLean/ccsvi-dashboard/public/data/Rasters/Hawaii_Category4_MOM_Inundation_HighTide.tif"
    r"C:/Users/Aaron McLean/ccsvi-dashboard/public/data/Rasters/lw_hi.tif"
)
info2

{'width': 8303,
 'height': 6233,
 'totalPixels': 51752599,
 'bands': 1,
 'dtype': 'int32',
 'bitsPerSample': 32,
 'bytesPerSample': 4,
 'compression': 'lzw',
 'isTiled': True,
 'tileShape': (128, 128),
 'noData': 0.0,
 'transform': Affine(0.000858460222819046, 0.0, -161.1145251034157,
        0.0, -0.0008584602228190461, 23.201557909244162),
 'crs': 'GEOGCS["NAD83",DATUM["North American Datum 1983",SPHEROID["GRS 1980",6378137,298.257222101004]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]'}

In [36]:
# for setting nodata 

import rasterio
with rasterio.open("C:/Users/Aaron McLean/ccsvi-dashboard/public/data/Rasters/lw_hi.tif", "r+") as src:
    src.nodata = 0
    # src.nodata = 2147483647

In [27]:
import sys
print(sys.executable)
print(sys.version)
import imagecodecs
print(imagecodecs.__version__)
!"{sys.executable}" -m pip install --upgrade imagecodecs

c:\Users\Aaron McLean\AppData\Local\Programs\Python\Python313\python.exe
3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
2025.11.11



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import rasterio
from rasterio.windows import Window

with rasterio.open(filename) as src:
    data = src.read(1, window=Window(0, 0, 5, 5))
    mask = src.read_masks(1, window=Window(0, 0, 5, 5))

    print("data:")
    print(data)
    print("mask:")
    print(mask)

data:
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
mask:
[[255 255 255 255 255]
 [255 255 255 255 255]
 [255 255 255 255 255]
 [255 255 255 255 255]
 [255 255 255 255 255]]


In [37]:
#for getting the first couple pixels to find nodata values
from tifffile import TiffFile
import imagecodecs
filename = "C:/Users/Aaron McLean/ccsvi-dashboard/public/data/Rasters/lw_hi.tif"

with TiffFile(filename) as tif:
    page = tif.pages[0]
    #this should have the header data
    tags = page.tags
    for tag in tags.values():
        print(tag.name)
        print(tag.value)
    # #this should be a 2D array with the raster data (arr[row][col])
    # arr = page.asarray()
    # print(arr[0][0])


ImageWidth
8303
ImageLength
6233
BitsPerSample
32
Compression
5
PhotometricInterpretation
1
SamplesPerPixel
1
PlanarConfiguration
1
Predictor
1
TileWidth
128
TileLength
128
TileOffsets
(26218, 27092, 27966, 28840, 29714, 30588, 31462, 32336, 33210, 34084, 34958, 35832, 36706, 37580, 38454, 39328, 40202, 41076, 41950, 42824, 43698, 44572, 45446, 46320, 47194, 48068, 48942, 49816, 50690, 51564, 52438, 53312, 54186, 55060, 55934, 56808, 57682, 58556, 59430, 60304, 61178, 62052, 62926, 63800, 64674, 65548, 66422, 67296, 68170, 69044, 69918, 70792, 71666, 72540, 73414, 74288, 75162, 76036, 76910, 77784, 78658, 79532, 80406, 81280, 82154, 83028, 83902, 84776, 85650, 86524, 87398, 88272, 89146, 90020, 90894, 91768, 92642, 93516, 94390, 95264, 96138, 97012, 97886, 98760, 99634, 100508, 101382, 102256, 103130, 104004, 104878, 105752, 106626, 107500, 108374, 109248, 110122, 110996, 111870, 112744, 113618, 114492, 115366, 116240, 117114, 117988, 118862, 119736, 120610, 121484, 122358, 123232, 124

In [42]:
filename = "C:/Users/Aaron McLean/ccsvi-dashboard/public/data/Rasters/Hawaii_Category4_MOM_Inundation_HighTide.tif"

with rasterio.open(filename) as src: 
    print("nodata:", src.nodata) 
    arr = src.read(1, window=Window(0, 0, 5, 5)) 
    print(arr)

nodata: 255.0
[[255 255 255 255 255]
 [255 255 255 255 255]
 [255 255 255 255 255]
 [255 255 255 255 255]
 [255 255 255 255 255]]
